## 04 · Comprobaciones preliminares de modelado

**Guía de ejecución:** Este notebook contiene análisis del corpus y pruebas de tokenización con BERT. NO corresponde al entrenamiento definitivo del clasificador MLP-MIL; ese experimento está en notebooks/colab/training_experiments.ipynb.

Instala las dependencias una vez desde la raíz del repositorio con `pip install -e ".[notebooks]"`. Los archivos de `data/` se mantienen fuera de Git. Este notebook se conserva para reproducir la fase de investigación; no se recomienda pulsar «Ejecutar todo» sin revisar sus fases y requisitos.


In [ ]:
# Preparación: iniciar Jupyter desde la raíz del repositorio (local o Colab).
from pathlib import Path
import os, sys

def _locate_repo():
    for folder in (Path.cwd(), *Path.cwd().parents, Path('/content/cognitive-bias-detector')):
        if (folder / 'pyproject.toml').is_file() and (folder / 'src' / 'bias_detector').is_dir():
            return folder.resolve()
    raise RuntimeError('No se encontró la raíz del repositorio. Abre Jupyter allí o usa %cd /content/cognitive-bias-detector en Colab.')

PROJECT_ROOT = _locate_repo()
source_path = str(PROJECT_ROOT / 'src')
if source_path not in sys.path:
    sys.path.insert(0, source_path)
from bias_detector.config import (
    DATA_DIR, MODEL_DATA_DIR, MODEL_DIR, PROMPTS_DIR, RESULTS_DIR,
    RAW_DIR, PROCESSED_DIR, MODELING_DIR, PILOT_DIR,
    REDACTION_ANALYSIS_DIR, ANNOTATION_RESULTS_DIR,
    MASS_ANNOTATION_DIR, FINAL_ANNOTATION_DIR, PILOT_RESULTS_DIR,
    MICRO_PILOT_RESULTS_DIR, CORPUS_DATA, FINAL_ANNOTATIONS_DATA,
    require_file, ensure_output_dir,
)
print('Repositorio:', PROJECT_ROOT)
print('Directorio de datos:', DATA_DIR)


### Importación de librerías

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoTokenizer


### Carga de datos

In [ ]:
PROJECT_ROOT = _locate_repo()

DATA_DIR = PROJECT_ROOT / "data" / "modeling"

df_corpus = pd.read_pickle(DATA_DIR / "modeling_dataset_v0_2.pkl")

In [ ]:
df_annotations = pd.read_pickle((PROJECT_ROOT / 'data' / 'results' / 'mass_annotation' / 'final' / 'final_bias_annotations_v0_2.pkl'))

In [ ]:
df_annotations.columns

In [ ]:
df_corpus.head(5)

In [ ]:
# ==============================================
# 1. CONFIGURACIÓN
# ==============================================

TEXT_COLUMN = "conversation_text"  # Ajustar si procede
MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# ==============================================
# 2. CÁLCULO DE TOKENS
# ==============================================

df_analysis = df_corpus.copy()

def count_tokens(text):
    """Cuenta tokens incluyendo los tokens especiales."""
    return len(
        tokenizer.encode(
            str(text),
            add_special_tokens=True,
            truncation=False
        )
    )

df_analysis["n_tokens"] = (
    df_analysis[TEXT_COLUMN].apply(count_tokens)
)

# Utilizamos las columnas ya existentes
columns = [
    "n_tokens",
    "turns_count_original",
    "n_messages"
]

df_analysis[["turns_count_original", "n_messages"]] = (
    df_analysis[["turns_count_original", "n_messages"]]
    .apply(pd.to_numeric, errors="coerce")
)

# ==============================================
# 3. ESTADÍSTICAS DESCRIPTIVAS
# ==============================================

statistics = df_analysis[columns].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

print("\n=== ESTADÍSTICAS DESCRIPTIVAS ===")
print(statistics.round(2).to_string())

# ==============================================
# 4. LÍMITES DE CONTEXTO
# ==============================================

print("\n=== LONGITUD DE CONVERSACIONES ===")

for limit in [128, 256, 512, 1024, 2048, 4096]:

    count = (df_analysis["n_tokens"] > limit).sum()
    percentage = count / len(df_analysis) * 100

    print(
        f"> {limit:4d} tokens: "
        f"{count:5d} conversaciones "
        f"({percentage:.2f}%)"
    )

# ==============================================
# 5. DISTRIBUCIONES
# ==============================================

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

variables = [
    ("n_tokens", "Tokens por conversación", 50),
    ("turns_count_original", "Turnos por conversación", 30),
    ("n_messages", "Mensajes por conversación", 30)
]

for ax, (column, label, bins) in zip(axes, variables):

    ax.hist(
        df_analysis[column].dropna(),
        bins=bins,
        edgecolor="black",
        alpha=0.75
    )

    ax.set_title(f"Distribución de {label.lower()}")
    ax.set_xlabel(label)
    ax.set_ylabel("Número de conversaciones")

# Referencia de longitud estándar de BERT
axes[0].axvline(
    512,
    color="red",
    linestyle="--",
    linewidth=2,
    label="Límite BERT (512 tokens)"
)

axes[0].legend()

plt.tight_layout()
plt.show()


In [ ]:

# Conversaciones con más turnos de los que permiten sus mensajes
inconsistent = df_corpus[
    df_corpus["turns_count_original"] * 2 >
    df_corpus["n_messages"]
]

print("Conversaciones potencialmente inconsistentes:", len(inconsistent))

display(
    inconsistent[
        ["conversation_id", "turns_count_original", "n_messages", "n_biases_present"]
    ].head(10)
)


In [ ]:
df_corpus.iloc[14]["conversation_text"]